
# Player Behavior Analytics from Poker Event Logs

This notebook is the **clean presentation layer** for the project.  
The reusable parsing, transformation, validation, and metric logic lives in `src/`.



## 1. Load and Normalize Raw Logs

Load raw JSON hand histories from `data/raw/` and build the core tables used for analysis.


In [1]:

from pathlib import Path
import sys

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.load_data import load_all_hands
from src.transform_data import build_core_tables

all_hands = load_all_hands(project_root / "data" / "raw")
actions_df, hands_df, players_df, hand_players_df = build_core_tables(all_hands)

print(f"Loaded {len(all_hands)} hands")
actions_df.head()


ModuleNotFoundError: No module named 'src'


## 2. Validation

Validate the structured event table before computing downstream metrics.


In [ ]:

from src.validate_data import validate_actions_df

validation_errors = validate_actions_df(actions_df)
validation_errors



## 3. Feature Engineering

Compute player-level metrics including VPIP, PFR, aggression factor, c-bet rate, and WTSD%.


In [ ]:

from src.build_metrics import build_all_metrics

final_df = build_all_metrics(actions_df, hand_players_df)
final_df.head(10)



## 4. Correlation Check


In [ ]:

final_df[["vpip", "preflop_raise_rate", "aggression_factor", "cbet_rate", "wtsd_percent"]].corr()



## 5. Visualization

A simple VPIP vs PFR scatter plot helps separate looser/passive profiles from tighter/aggressive ones.


In [ ]:

import matplotlib.pyplot as plt

plot_df = final_df.copy()

plt.figure(figsize=(8, 6))
plt.scatter(plot_df["vpip"], plot_df["preflop_raise_rate"])

for _, row in plot_df.iterrows():
    plt.text(row["vpip"], row["preflop_raise_rate"], row["player_name"], fontsize=8)

plt.xlabel("VPIP")
plt.ylabel("Preflop Raise Rate")
plt.title("VPIP vs PFR by Player")
plt.tight_layout()
plt.show()



## 6. Key Takeaways

- Players with high VPIP and high PFR tend to show looser, more aggressive styles.
- High VPIP with low PFR can indicate passive participation.
- Combining preflop and postflop metrics gives a more complete picture than any single stat alone.
